In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf

In [ ]:

# CONSTANTS

# Initial investment capital for each client in CAD
budget_A = 1000000
budget_B = 750000

# Portfolio restrictions provided in the assignment
MIN_STOCKS = 10
MAX_STOCKS = 20
MAX_WEIGHT = 0.15
MAX_SECTOR_WEIGHT = 0.40

# Transaction fee restrictions in USD
FEE_CAP_USD = 2.15
FEE_PER_SHARE_USD = 0.001

# Required purchase date and group number
PURCHASE_DATE = "2026-11-27"
GROUP_NUMBER = "XX"

# Tolerance accounts for small floating point calculation errors
TOL = 0.000001

# CLIENT A - GROWTH PORTFOLIO


# Find the most recent date with available growth indicators
available_dates = high_return_tables["adx 3M"].dropna(how="all").index

latest = available_dates[available_dates <= PURCHASE_DATE][-1]

# Identify stocks that pass the optimized growth strategy filters
passed = passes(high_return_tables, HIGH_RET_RULES, best_hr, latest)

# Select stocks using the existing growth strategy
growth_picks = select_growth_stocks(latest, passed)

# Calculate target weights using the growth scoring method
# The existing function also applies stock and sector weight limits
final_weights = final_growth_weights(growth_picks, latest)

# Organize the selected stocks, sectors and weights into a DataFrame
growth_portfolio = pd.DataFrame({
    "Ticker": final_weights.index,
    "Sector": [stock_sectors[t] for t in final_weights.index],
    "Weight": final_weights.values})


# CLIENT B - LOW RISK PORTFOLIO

# Only consider stocks with sufficient historical data
# and available sector information
known_sector_universe = [
    t for t in usable_universe(lookback_returns(latest))
    if pd.notna(stock_sectors.get(t))
    and stock_sectors.get(t) != ""
]

# Select stocks using the existing low-volatility strategy
low_vol_picks = build_low_vol(
    latest, universe=known_sector_universe
)

# Calculate portfolio weights using the selected stocks
# and the latest available historical date
low_vol_portfolio = make_low_vol_portfolio(
    low_vol_picks, latest
)


In [ ]:

# RETRIEVE CLOSING PRICES

def get_close(symbol, date):

    # Convert the purchase date into a pandas date
    target = pd.Timestamp(date)

    # Download a week of price data in case the requested
    # date falls on a holiday or is not yet available
    history = yf.Ticker(symbol).history(
        start=target - pd.Timedelta(days=7),
        end=target + pd.Timedelta(days=1),
        auto_adjust=False )

    # Remove missing or invalid closing prices
    closing_prices = history["Close"].dropna()
    closing_prices = closing_prices[
        np.isfinite(closing_prices) & (closing_prices > 0)]

    # Stop if no valid price data is available
    if closing_prices.empty:
        raise ValueError("Missing price data for " + symbol)

    # Identify the date of the most recent available price
    actual_date = closing_prices.index[-1].strftime("%Y-%m-%d")

    # Notify the user if the required closing price is unavailable
    if actual_date != date:
        print(symbol, "using latest available price:", actual_date)

    # Return the latest available price in its original currency
    return float(closing_prices.iloc[-1])


# EXCHANGE RATE

# CADUSD represents the number of USD received for one CAD
# Dividing a USD amount by this rate converts it into CAD
purchase_cadusd = get_close("CADUSD=X", PURCHASE_DATE)

print("CAD to USD exchange rate:", purchase_cadusd)


# TRANSACTION FEE CALCULATION

def calculate_fees(shares, exchange_rate):

    # The transaction fee is the smaller of:
    # 1. USD $2.15 per stock purchased
    # 2. USD $0.001 multiplied by the number of shares purchased

    fees_usd = np.minimum(
        FEE_CAP_USD,
        FEE_PER_SHARE_USD * shares)

    # All transaction fees are charged in USD,
    # even when purchasing Canadian stocks

    # Convert the transaction fees into CAD
    fees_cad = fees_usd / exchange_rate

    return fees_cad

In [ ]:

# FINAL PORTFOLIO CONSTRUCTION 

def build_portfolio(portfolio, budget, exchange_rate):

    # Create a copy so the original strategy portfolio is unchanged
    portfolio = portfolio.copy()

    # Check that the portfolio contains at least one stock
    if portfolio.empty:
        raise ValueError("Portfolio is empty")

    # Ensure the portfolio does not contain duplicate stocks
    if not portfolio["Ticker"].is_unique:
        raise ValueError("Duplicate tickers")

    # Ensure all target weights are finite and positive
    # This prevents negative holdings and short selling
    if not (
        np.isfinite(portfolio["Weight"])
        & (portfolio["Weight"] > 0)
    ).all():
        raise ValueError("Invalid portfolio weights")

    # Check that target weights add up to 100%
    if abs(portfolio["Weight"].sum() - 1) > TOL:
        raise ValueError("Target weights must sum to 100%")

    # Normalize weights to remove small floating point errors
    portfolio["Weight"] = (
        portfolio["Weight"] / portfolio["Weight"].sum()
    )


    # RETRIEVE PRICES AND CURRENCIES

    prices = []
    currencies = []

    for ticker in portfolio["Ticker"]:

        # Retrieve the closing price for the purchase date
        prices.append(get_close(ticker, PURCHASE_DATE))

        # Retrieve currency from the initial stock filtering section
        currencies.append(stock_info[ticker]["currency"])

    portfolio["Price"] = prices
    portfolio["Currency"] = currencies


    # CONVERT STOCK PRICES INTO CAD

    price_cad = []

    for i in range(len(portfolio)):

        price = portfolio["Price"].iloc[i]
        currency = portfolio["Currency"].iloc[i]

        # Convert American stock prices from USD into CAD
        if currency == "USD":
            price_cad.append(price / exchange_rate)

        # Canadian stocks do not require currency conversion
        elif currency == "CAD":
            price_cad.append(price)

        # Prevent purchases of stocks with unsupported currencies
        else:
            raise ValueError("Invalid stock currency")

    portfolio["Price_CAD"] = price_cad



    # DETERMINE INVESTABLE CAPITAL AFTER FEES

    # Transaction fees depend on the number of shares purchased.
    # However, the number of shares purchased also depends on
    # how much money remains after paying transaction fees.
    # Binary search finds the largest affordable investment amount
    # without exceeding the client's total budget.

    lower = 0
    upper = budget

    for i in range(60):

        # Test an investment amount halfway between the two limits
        investment = (lower + upper) / 2

        # Allocate the investment according to the target weights
        # Divide each allocation by its CAD stock price to get shares
        shares = (
            investment * portfolio["Weight"]
            / portfolio["Price_CAD"]
        )

        # Calculate the transaction fees for these purchases
        fees = calculate_fees(shares, exchange_rate)

        # Calculate the total cost of shares and transaction fees
        total_spent = (
            (shares * portfolio["Price_CAD"]).sum()
            + fees.sum()
        )

        # If spending exceeds the budget, reduce the upper limit
        if total_spent > budget:
            upper = investment

        # Otherwise, a larger investment may still be affordable
        else:
            lower = investment

    # Use the largest affordable investment amount found
    investment = lower

    # CALCULATE FINAL HOLDINGS

    # Calculate fractional shares purchased for each stock
    portfolio["Shares"] = (
        investment * portfolio["Weight"]
        / portfolio["Price_CAD"])

    # Calculate the total value of each stock holding in CAD
    portfolio["Value"] = (
        portfolio["Shares"] * portfolio["Price_CAD"])

    # Calculate actual weights based on total invested value
    # Transaction fees are not included in portfolio market value
    portfolio["Weight"] = (portfolio["Value"] / portfolio["Value"].sum())


    
    # FORMAT FINAL PORTFOLIO

    # Keep only the six columns required by the assignment
    final = portfolio[
        ["Ticker", "Price", "Currency", "Shares", "Value", "Weight"]
    ].copy()

    # Change the index so it starts at 1 instead of 0
    final.index = range(1, len(final) + 1)

    return final

# CREATE FINAL PORTFOLIOS FOR BOTH CLIENTS

Portfolio_Final_A = build_portfolio(
    growth_portfolio,
    budget_A,
    purchase_cadusd)

Portfolio_Final_B = build_portfolio(
    low_vol_portfolio,
    budget_B,
    purchase_cadusd)

In [ ]:
# PROGRAMMATIC VALIDATION - task 3

# Retrieve the original list of tickers from the supplied CSV
# Remove extra spaces and standardize tickers to uppercase

tickers_file_list = []

for ticker in tickers.iloc[:, 0]:

    tickers_file_list.append(
        str(ticker).strip().upper()
    )


def validate_portfolio(portfolio, budget, exchange_rate):

    # Store the name, result and details of each validation test
    tests = []


    
    # TEST 1: VALID AND ELIGIBLE TICKERS
    

    # Check that every selected stock appears in the supplied CSV
    in_ticker_file = portfolio["Ticker"].isin(
        tickers_file_list
    ).all()

    # Check that every stock passed the initial eligibility filter
    # This includes company stock, market, currency and volume rules
    in_eligible_list = portfolio["Ticker"].isin(
        eligible
    ).all()

    # Ensure the same stock is not included more than once
    no_duplicates = portfolio["Ticker"].is_unique

    # All three conditions must be satisfied
    valid_tickers = (
        in_ticker_file
        and in_eligible_list
        and no_duplicates
    )

    tests.append([
        "Valid and eligible tickers",
        valid_tickers,
        f"In file: {in_ticker_file}, Eligible: {in_eligible_list}, "
        f"Unique: {no_duplicates}"
    ])


    
    # TEST 2: STOCK COUNT

    # Count the number of stocks in the final portfolio
    stock_count = len(portfolio)

    # The portfolio must contain between 10 and 20 stocks
    tests.append([
        "Stock count (10-20)",
        MIN_STOCKS <= stock_count <= MAX_STOCKS,
        f"{stock_count} stocks"
    ])


    
    # TEST 3: MAXIMUM INDIVIDUAL STOCK WEIGHT

    # Find the stock with the largest portfolio weight
    largest_weight = portfolio["Weight"].max()

    # Check that no individual stock exceeds the 15% limit
    tests.append([
        "Maximum stock weight (15%)",
        largest_weight <= MAX_WEIGHT + TOL,
        f"{largest_weight:.4%}"
    ])


    # TEST 4: MAXIMUM SECTOR WEIGHT
    
    # Create a temporary copy to include sector information
    sector_portfolio = portfolio.copy()

    # Match each stock to its sector using the existing dictionary
    sector_portfolio["Sector"] = sector_portfolio[
        "Ticker"
    ].map(stock_sectors)

    # Ensure sector information is available for every stock
    known_sectors = (
        sector_portfolio["Sector"].notna().all()
        and (sector_portfolio["Sector"] != "").all()
    )

    # Group stocks by sector and add their portfolio weights
    sector_weights = calculate_sector_weights(
        sector_portfolio
    )

    # Find the sector with the highest combined weight
    largest_sector_weight = sector_weights.max()

    # Check that no sector exceeds 40% of the portfolio
    tests.append([
        "Maximum sector weight (40%)",
        known_sectors and largest_sector_weight <= MAX_SECTOR_WEIGHT + TOL,
        f"{largest_sector_weight:.4%}"
    ])


    # TEST 5: TOTAL PORTFOLIO WEIGHT

    # Add together all individual stock weights
    total_weight = portfolio["Weight"].sum()

    # Check that the total weight equals 100%
    # Tolerance is used for small floating point errors
    tests.append([
        "Weights sum to 100%",
        abs(total_weight - 1) <= TOL,
        f"{total_weight:.4%}"
    ])


    # TEST 6: TOTAL PURCHASES AND FEES WITHIN BUDGET

    # Calculate the total amount invested in stocks
    purchases = portfolio["Value"].sum()

    # Recalculate fees using the actual number of shares purchased
    transaction_fees = calculate_fees(
        portfolio["Shares"],
        exchange_rate
    ).sum()

    # Calculate total spending, including all transaction fees
    total_spent = purchases + transaction_fees

    # Confirm that the client's initial budget is not exceeded
    tests.append([
        "Purchases + fees within budget",
        total_spent <= budget + TOL,
        f"${total_spent:,.2f} / ${budget:,.2f}"])

    
    # ORGANIZE VALIDATION RESULTS

    # Convert the list of validation tests into a DataFrame
    validation = pd.DataFrame(
        tests,
        columns=["Test", "Result", "Details"])

    # Replace Boolean values with readable PASS or FAIL results
    validation["Result"] = validation["Result"].map({
        True: "PASS",
        False: "FAIL"})

    return validation


# VALIDATE BOTH CLIENT PORTFOLIOS

Validation_A = validate_portfolio(
    Portfolio_Final_A,
    budget_A,
    purchase_cadusd)

Validation_B = validate_portfolio(
    Portfolio_Final_B,
    budget_B,
    purchase_cadusd)


# Display validation results for both clients

print("CLIENT A - VALIDATION")
display(Validation_A)

print("CLIENT B - VALIDATION")
display(Validation_B)

In [ ]:
# BUDGET RECONCILIATION

def budget_reconciliation(portfolio, budget, exchange_rate):

    # Calculate the total CAD value of purchased stocks
    purchases = portfolio["Value"].sum()

    # Calculate the total transaction fees in CAD
    transaction_fees = calculate_fees(
        portfolio["Shares"],
        exchange_rate
    ).sum()

    # Calculate total spending, including transaction fees
    total_spent = purchases + transaction_fees

    # Calculate the remaining unused investment capital
    remaining_cash = budget - total_spent

    # Organize the budget calculations into a DataFrame
    reconciliation = pd.DataFrame({
        "Description": [
            "Initial Budget",
            "Stock Purchases",
            "Transaction Fees",
            "Total Spent",
            "Remaining Cash"
        ],

        "Amount (CAD)": [
            budget,
            purchases,
            transaction_fees,
            total_spent,
            remaining_cash
        ]
    })

    return reconciliation


# CREATE BUDGET RECONCILIATION TABLES

Budget_A = budget_reconciliation(
    Portfolio_Final_A,
    budget_A,
    purchase_cadusd
)

Budget_B = budget_reconciliation(
    Portfolio_Final_B,
    budget_B,
    purchase_cadusd
)


# CLIENT A - FINAL PORTFOLIO

print("CLIENT A - GROWTH PORTFOLIO")

# Format the displayed values without changing the actual data
display(Portfolio_Final_A.style.format({
    "Price": "{:,.2f}",
    "Shares": "{:,.6f}",
    "Value": "${:,.2f}",
    "Weight": "{:.2%}"
}))

# Display purchases, transaction fees and remaining cash
print("CLIENT A - BUDGET RECONCILIATION")

display(Budget_A.style.format({
    "Amount (CAD)": "${:,.2f}"
}))


# CLIENT B - FINAL PORTFOLIO

print("CLIENT B - LOW RISK PORTFOLIO")

# Display the final holdings for Client B
display(Portfolio_Final_B.style.format({
    "Price": "{:,.2f}",
    "Shares": "{:,.6f}",
    "Value": "${:,.2f}",
    "Weight": "{:.2%}"
}))

# Display purchases, transaction fees and remaining cash
print("CLIENT B - BUDGET RECONCILIATION")

display(Budget_B.style.format({
    "Amount (CAD)": "${:,.2f}"
}))


# GENERATE REQUIRED CSV FILES

# Check whether both portfolios passed all six validation tests
passed_A = (Validation_A["Result"] == "PASS").all()
passed_B = (Validation_B["Result"] == "PASS").all()

# Only generate the CSV files if both portfolios are valid
if passed_A and passed_B:

    # Export Client A's tickers and fractional shares
    Portfolio_Final_A[["Ticker", "Shares"]].to_csv(
        f"Stocks_Group_{GROUP_NUMBER}_A.csv",
        index=False
    )

    # Export Client B's tickers and fractional shares
    Portfolio_Final_B[["Ticker", "Shares"]].to_csv(
        f"Stocks_Group_{GROUP_NUMBER}_B.csv",
        index=False
    )

    print("Both portfolios passed all six validation tests.")
    print("Both CSV files generated successfully.")

else:

    # Identify which portfolio failed validation
    if not passed_A:
        print("Client A failed validation.")

    if not passed_B:
        print("Client B failed validation.")

    print("Review the validation tables before exporting.") 